# Cálculo de Métricas HDBSCAN
Este notebook calcula la Homogeneidad y Pureza de los clusters formados por HDBSCAN basándose en las etiquetas verdaderas del dataset provisto en formato JSON.

In [1]:
import os
import re
import json
from collections import defaultdict, Counter
from sklearn.metrics import homogeneity_score

# --- CONFIGURACIÓN ---
CLUSTERS_DIR = "Clusters"
JSON_PATH = "island_conservation_camera_traps_1.02.json"
# ---------------------

In [2]:
print("1. Buscando recortes en las carpetas de clusters...")
crop_files = []

if not os.path.exists(CLUSTERS_DIR):
    print(f"Error: No se encontró el directorio {CLUSTERS_DIR}")
else:
    for root, dirs, files in os.walk(CLUSTERS_DIR):
        cluster_name = os.path.basename(root)
        if cluster_name == "Clusters": continue
        
        for file in files:
            if file.lower().endswith('.jpg'):
                match = re.search(r'(.*)_crop(\d+)\.jpg$', file, re.IGNORECASE)
                if match:
                    basename = match.group(1)
                    crop_idx = int(match.group(2))
                    crop_files.append({
                        'cluster': cluster_name,
                        'basename': basename,
                        'crop_idx': crop_idx,
                        'filename': file
                    })

    print(f"Se encontraron {len(crop_files)} recortes agrupados en carpetas.")

    unique_basenames = {item['basename'] for item in crop_files}

    print(f"2. Cargando JSON de anotaciones desde {JSON_PATH}...")
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)

    print("3. Procesando listado de anotaciones...")
    img_id_to_annotations = defaultdict(list)
    for ann in data.get("annotations", []):
        img_id_to_annotations[ann["image_id"]].append(ann["category_id"])

    print("4. Emparejando archivos físicos con IDs del JSON...")
    basename_to_image_id = {}
    for img_id in img_id_to_annotations.keys():
        for b_name in unique_basenames:
            if img_id.endswith(b_name) or img_id == b_name:
                basename_to_image_id[b_name] = img_id
                
    print("5. Preparando listas para métricas de Homogeneidad y Pureza...")
    y_true_all = []
    y_pred_all = []
    y_true_no_noise = []
    y_pred_no_noise = []
    cluster_contents = defaultdict(list)

    for crop in crop_files:
        basename = crop['basename']
        cluster = crop['cluster']
        crop_idx = crop['crop_idx']
        
        img_id = basename_to_image_id.get(basename)
        if img_id is None: continue
            
        categories = img_id_to_annotations[img_id]
        if crop_idx < len(categories):
            cat_id = categories[crop_idx]
            y_true_all.append(cat_id)
            y_pred_all.append(cluster)
            
            is_noise_cluster = str(cluster) in ["-1", "Cluster_-1", "Cluster_Ruido_Outliers"]
            if not is_noise_cluster:
                y_true_no_noise.append(cat_id)
                y_pred_no_noise.append(cluster)
            
            cluster_contents[cluster].append(cat_id)


1. Buscando recortes en las carpetas de clusters...
Se encontraron 1653 recortes agrupados en carpetas.
2. Cargando JSON de anotaciones desde island_conservation_camera_traps_1.02.json...
3. Procesando listado de anotaciones...
4. Emparejando archivos físicos con IDs del JSON...
5. Preparando listas para métricas de Homogeneidad y Pureza...


In [3]:
if not y_true_all:
    print("Error: No se logró completar el emparejamiento con el JSON.")
else:
    homogeneity_all = homogeneity_score(y_true_all, y_pred_all)
    
    print("\n" + "=" * 50)
    print("RESULTADOS GLOBALES (Homogeneidad)")
    print("=" * 50)
    print(f"Total imágenes analizadas exitosamente: {len(y_true_all)}")
    print(f"Homogeneidad Global (Con cluster -1): {homogeneity_all:.4f}")
    
    if len(y_true_no_noise) > 0 and len(y_true_no_noise) < len(y_true_all):
        homog_no_noise = homogeneity_score(y_true_no_noise, y_pred_no_noise)
        print(f"Homogeneidad Global (Sin cluster -1): {homog_no_noise:.4f}")
        print("(El cluster -1 o ruido empeora la métrica global, lo ideal es guiarte por este segundo número)")

    print("\n" + "=" * 50)
    print("PUREZA POR CLUSTER (Purity)")
    print("=" * 50)
    
    total_images_processed = 0
    total_correct_predictions = 0
    total_images_no_noise = 0
    total_correct_no_noise = 0
    
    def sort_cluster_key(k):
        if str(k) in ["-1", "Cluster_-1", "Cluster_Ruido_Outliers"]: return 999999
        try: return int(str(k).replace("Cluster_", ""))
        except ValueError: return str(k)

    for cluster in sorted(cluster_contents.keys(), key=sort_cluster_key):
        labels = cluster_contents[cluster]
        if not labels: continue
            
        counter = Counter(labels)
        most_common_cat, max_count = counter.most_common(1)[0]
        total_in_cluster = len(labels)
        purity = max_count / total_in_cluster
        
        total_images_processed += total_in_cluster
        total_correct_predictions += max_count
        
        is_noise_cluster = str(cluster) in ["-1", "Cluster_-1", "Cluster_Ruido_Outliers"]
        if not is_noise_cluster:
            total_images_no_noise += total_in_cluster
            total_correct_no_noise += max_count
        
        print(f"  [{str(cluster)[:25]:^25}]  ->  Pureza: {purity*100:6.2f}%  (Especie {most_common_cat} domina con {max_count}/{total_in_cluster} recortes)")

    if total_images_processed > 0:
        global_purity_all = total_correct_predictions / total_images_processed
        print("-" * 50)
        print(f"  Pureza Global (CON Ruido/Outliers):       {global_purity_all*100:.2f}%")
        
        if total_images_no_noise > 0 and total_images_no_noise < total_images_processed:
            global_purity_no_noise = total_correct_no_noise / total_images_no_noise
            print(f"  Pureza Global (SIN tomar en cuenta Ruido): {global_purity_no_noise*100:.2f}%")
            
        print("=" * 50)



RESULTADOS GLOBALES (Homogeneidad)
Total imágenes analizadas exitosamente: 1638
Homogeneidad Global (Con cluster -1): 0.4617
Homogeneidad Global (Sin cluster -1): 0.6141
(El cluster -1 o ruido empeora la métrica global, lo ideal es guiarte por este segundo número)

PUREZA POR CLUSTER (Purity)
  [       Cluster_00        ]  ->  Pureza: 100.00%  (Especie 5 domina con 23/23 recortes)
  [       Cluster_01        ]  ->  Pureza:  74.58%  (Especie 32 domina con 44/59 recortes)
  [       Cluster_02        ]  ->  Pureza: 100.00%  (Especie 6 domina con 29/29 recortes)
  [       Cluster_03        ]  ->  Pureza:  95.83%  (Especie 21 domina con 23/24 recortes)
  [       Cluster_04        ]  ->  Pureza:  95.45%  (Especie 12 domina con 21/22 recortes)
  [       Cluster_05        ]  ->  Pureza:  76.47%  (Especie 0 domina con 13/17 recortes)
  [       Cluster_06        ]  ->  Pureza: 100.00%  (Especie 7 domina con 30/30 recortes)
  [       Cluster_07        ]  ->  Pureza:  93.33%  (Especie 32 domina c